### Базовая подготовка

In [ ]:
import torch
import torchvision


print(f"Torch: {torch.__version__}")
print(f"Cuda_available: {torch.cuda.is_available()}")
print(f"Torchvision: {torchvision.__version__}")
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"{device = }")

### Предполагаемые метрики для оценки моделей

In [ ]:
# TODO: отобрать нужные
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

In [ ]:
import os
import rioxarray as rxr
import matplotlib.pyplot as plt
from terratorch.registry import FULL_MODEL_REGISTRY
from plotting_utils import plot_s2, plot_modality

In [ ]:
all_modalities = ['S2L2A', 'S1GRD', 'S1RTC', 'DEM', 'LULC', 'NDVI']
modality = 'LULC'

output_modalities = all_modalities[:]
output_modalities.remove(modality)

In [ ]:
def build_model_from_registry(
        name: str = 'terramind_v1_base_generate',
        input_modalities: list = ['S2L2A'],
        output_modalities: list = ['LULC'],
        pretrained: bool = True,
        standartize: bool = True,
        ):
    model = FULL_MODEL_REGISTRY.build(
    name,
    modalities=input_modalities,
    output_modalities=output_modalities,
    pretrained=pretrained,
    standardize=standartize,
    )

    return model

In [ ]:
model = build_model_from_registry(
    input_modalities=[modality],
    output_modalities=output_modalities
)

model = model.to(device)   

In [ ]:

examples = []
examples_dir_name = os.getcwd() + os.sep + 'examples' + os.sep + modality
if os.path.exists(examples_dir_name):
  examples = [examples_dir_name + os.sep + filename for filename in os.listdir(examples_dir_name)]

In [ ]:
def tif_to_tensor(filename: str):
    tif = rxr.open_rasterio(filename)
    # Convert to shape [B, C, 224, 224]
    image_tensor = torch.Tensor(tif.values, device='cpu').unsqueeze(0)
    return image_tensor

In [ ]:
example_id = 0

data = tif_to_tensor(examples[example_id])

In [ ]:
plot_modality(modality, data)

In [ ]:
def generate(
        model,
        input_data: list[torch.Tensor],
        verbose: bool = False,
        timesteps: int = 10,
) -> list:
    generated = []

    for data_instance in input_data:
        data = data_instance.to(device)
        with torch.no_grad():
            generated.append(model(data, verbose=verbose, timesteps=timesteps))

    return generated
    

In [ ]:
generated = generate(model, input_data=[data])[0]

In [ ]:
n_plots = len(generated) + 1
fig, ax = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))

plot_modality(modality, data, ax=ax[0])
ax[0].set_title('Input')

for i, (mod, value) in enumerate(generated.items()):
    plot_modality(mod, value, ax=ax[i + 1])

    ax[i+1].set_title('generated ' + mod)
    
plt.show()